# Brinkman stabilization and Oseen transport

Check affine velocity and pressure under increasing drag, including every residual term in the P1/P1 USFEM form. A separate Oseen patch verifies the skew-advection and Robin sign conventions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace, solve_darcy, solve_brinkman, solve_elasticity


In [ ]:
def velocity(x):
    """Return a rigid rotation with zero divergence."""
    return np.column_stack((x[:,1],-x[:,0]))
mesh=TriangleMesh.unit_square(2)
space=SkeletonSpace(mesh,tuple(FaceSpace.uniform(1) for _ in mesh.faces),2)
for drag in (0.,1.,100.):
    result=solve_brinkman(mesh,drag=drag,source=lambda x,d=drag:d*velocity(x)+np.array([1.,2.]),dirichlet=velocity,skeleton=space,formulation='usfem',local_refinement=4)
    error=result.pressure_l2_error(lambda x:x[:,0]+2*x[:,1]-1.5)
    assert result.l2_error(velocity)<1e-10 and error<1e-9
    print('drag',drag,'pressure error',error)
oseen=solve_brinkman(mesh,drag=1,advection=(2.,3.),source=lambda x:velocity(x)+[3.,-2.],dirichlet=velocity,skeleton=space)
assert oseen.l2_error(velocity)<1e-10


This is a polynomial consistency check, not a proof of uniform robustness in unresolved Brinkman or convection boundary layers. The native drag and background advection are constant; tensor/variable local coefficients may be expressed through UFL.